# Week 6: Feature Engineering, Market Metrics, and School District Enrichment

This notebook transforms the Week 5 analysis-ready MLS datasets into dashboard-ready datasets for market analysis and Tableau development.

## Objectives

- Prepare required date and numeric fields
- Create pricing, time-series, and transaction-timeline features
- Validate engineered metrics without hiding source-data issues
- Map properties to California Unified School Districts
- Create segmented market and competitive summary tables
- Export enriched datasets and quality-assurance outputs

> The original fields and raw calculations are retained. Validation flags and analysis-ready versions are added separately so that data-quality decisions remain transparent.

## Part 1: Environment and Project Paths

Import the required libraries and define reusable project paths. This notebook is designed to run from the project's `script` folder or from the project root.

If GeoPandas is not installed in the active environment, run `%pip install geopandas requests` in a separate cell and restart the kernel before continuing.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import geopandas as gpd
    import requests
except ImportError as exc:
    raise ImportError(
        "School district enrichment requires geopandas and requests. "
        "Run `%pip install geopandas requests`, restart the kernel, and rerun this cell."
    ) from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name.lower() in {"script", "scripts", "notebook", "notebooks"}
    else CURRENT_DIR
)
INPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Input directory:", INPUT_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

## Part 2: Load Analysis-Ready Datasets

Load the Listings and Sold datasets exported during Week 5. Required files are checked before loading so missing-path problems produce a clear message.

In [ ]:
listings_path = INPUT_DIR / "analysis_ready_listings.csv"
sold_path = INPUT_DIR / "analysis_ready_sold.csv"

missing_files = [str(path) for path in (listings_path, sold_path) if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Required Week 5 input files were not found: {missing_files}")

listings = pd.read_csv(listings_path, low_memory=False)
sold = pd.read_csv(sold_path, low_memory=False)

load_summary = pd.DataFrame({
    "Dataset": ["Listings", "Sold"],
    "Rows": [len(listings), len(sold)],
    "Columns": [listings.shape[1], sold.shape[1]],
    "DuplicateRows": [listings.duplicated().sum(), sold.duplicated().sum()],
})
display(load_summary)

## Part 3: Prepare Date, Numeric, and Coordinate Fields

CSV files do not preserve Pandas datetime types. Required transaction dates are converted back to datetime, and numeric fields are standardized using `errors="coerce"` so invalid source values become missing rather than causing silent string operations.

In [ ]:
date_columns = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
]

numeric_columns = [
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket",
    "Latitude",
    "Longitude",
]

for dataset_name, df in {"Listings": listings, "Sold": sold}.items():
    for column in date_columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")

    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    print(f"{dataset_name}: date and numeric preparation completed.")

required_sold_columns = [
    "CloseDate", "PurchaseContractDate", "ListingContractDate",
    "ClosePrice", "OriginalListPrice", "LivingArea", "DaysOnMarket",
]
missing_required = [column for column in required_sold_columns if column not in sold.columns]
if missing_required:
    raise KeyError(f"Sold dataset is missing required columns: {missing_required}")

## Part 4: Create Pricing and Negotiation Features

### Metrics

- **Price Ratio** = ClosePrice / OriginalListPrice
- **Close-to-Original-List Ratio** = ClosePrice / OriginalListPrice
- **Price Per Square Foot** = ClosePrice / LivingArea

The two ratio names are retained because the project handbook and dashboard may use both terms. Division is performed only when the denominator is positive.

In [ ]:
valid_original_price = sold["OriginalListPrice"].gt(0) & sold["ClosePrice"].notna()
valid_living_area = sold["LivingArea"].gt(0) & sold["ClosePrice"].notna()

sold["PriceRatio"] = np.where(
    valid_original_price,
    sold["ClosePrice"] / sold["OriginalListPrice"],
    np.nan,
)
sold["CloseToOriginalListRatio"] = sold["PriceRatio"]

sold["PricePerSqFt"] = np.where(
    valid_living_area,
    sold["ClosePrice"] / sold["LivingArea"],
    np.nan,
)

pricing_metrics = ["PriceRatio", "CloseToOriginalListRatio", "PricePerSqFt"]
display(sold[pricing_metrics].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).T)

## Part 5: Create Time-Series Features

Sold-property trends use `CloseDate`, while listing activity uses `ListingContractDate`. Month numbers are retained for filtering, and year-month values are stored in `YYYY-MM` format for Tableau time-series views.

In [ ]:
sold["Year"] = sold["CloseDate"].dt.year.astype("Int64")
sold["Month"] = sold["CloseDate"].dt.month.astype("Int64")
sold["YrMo"] = sold["CloseDate"].dt.to_period("M").astype("string")

if "ListingContractDate" not in listings.columns:
    raise KeyError("Listings dataset requires ListingContractDate for listing time features.")

listings["ListingYear"] = listings["ListingContractDate"].dt.year.astype("Int64")
listings["ListingMonth"] = listings["ListingContractDate"].dt.month.astype("Int64")
listings["ListingYrMo"] = listings["ListingContractDate"].dt.to_period("M").astype("string")

display(sold[["CloseDate", "Year", "Month", "YrMo"]].head())
display(listings[["ListingContractDate", "ListingYear", "ListingMonth", "ListingYrMo"]].head())

## Part 6: Create Transaction-Timeline Features

- **Listing-to-Contract Days** = PurchaseContractDate − ListingContractDate
- **Contract-to-Close Days** = CloseDate − PurchaseContractDate

Raw values are preserved for QA. Separate valid versions exclude negative durations from market summaries without deleting the underlying records.

In [ ]:
sold["ListingToContractDays"] = (
    sold["PurchaseContractDate"] - sold["ListingContractDate"]
).dt.days

sold["ContractToCloseDays"] = (
    sold["CloseDate"] - sold["PurchaseContractDate"]
).dt.days

sold["ListingToContractDaysValid"] = sold["ListingToContractDays"].where(
    sold["ListingToContractDays"].ge(0)
)
sold["ContractToCloseDaysValid"] = sold["ContractToCloseDays"].where(
    sold["ContractToCloseDays"].ge(0)
)

timeline_metrics = [
    "DaysOnMarket",
    "ListingToContractDays",
    "ListingToContractDaysValid",
    "ContractToCloseDays",
    "ContractToCloseDaysValid",
]
display(sold[timeline_metrics].describe(percentiles=[0.01, 0.50, 0.99]).T)

## Part 7: Validate Engineered Metrics and Flag Extreme Values

The source data contains extreme values that can distort averages. This notebook preserves the original calculations and creates transparent validation flags instead of silently deleting records.

Broad business-review thresholds are used only to identify records requiring investigation:

- Price Ratio outside 0.50–1.50
- Price Per Square Foot outside $50–$5,000
- Days on Market below 0 or above 1,095 days

These thresholds should be reviewed with the project stakeholder before any permanent exclusion rule is adopted.

In [ ]:
sold["PriceRatioReviewFlag"] = sold["PriceRatio"].notna() & ~sold["PriceRatio"].between(0.50, 1.50)
sold["PricePerSqFtReviewFlag"] = sold["PricePerSqFt"].notna() & ~sold["PricePerSqFt"].between(50, 5000)
sold["DaysOnMarketReviewFlag"] = sold["DaysOnMarket"].notna() & ~sold["DaysOnMarket"].between(0, 1095)
sold["TimelineReviewFlag"] = (
    sold["ListingToContractDays"].lt(0).fillna(False)
    | sold["ContractToCloseDays"].lt(0).fillna(False)
)
sold["AnyEngineeredMetricReviewFlag"] = sold[[
    "PriceRatioReviewFlag",
    "PricePerSqFtReviewFlag",
    "DaysOnMarketReviewFlag",
    "TimelineReviewFlag",
]].any(axis=1)

quality_summary = pd.DataFrame({
    "Check": [
        "Price ratio requires review",
        "Price per square foot requires review",
        "Days on market requires review",
        "Transaction timeline requires review",
        "Any engineered metric requires review",
    ],
    "Count": [
        sold["PriceRatioReviewFlag"].sum(),
        sold["PricePerSqFtReviewFlag"].sum(),
        sold["DaysOnMarketReviewFlag"].sum(),
        sold["TimelineReviewFlag"].sum(),
        sold["AnyEngineeredMetricReviewFlag"].sum(),
    ],
})
quality_summary["PercentOfSoldRecords"] = quality_summary["Count"] / len(sold)
display(quality_summary)

In [ ]:
engineered_columns = [
    "ListingKey", "ClosePrice", "OriginalListPrice", "PriceRatio",
    "LivingArea", "PricePerSqFt", "DaysOnMarket", "CloseDate", "YrMo",
    "ListingToContractDaysValid", "ContractToCloseDaysValid",
    "AnyEngineeredMetricReviewFlag",
]
available_engineered_columns = [column for column in engineered_columns if column in sold.columns]
display(sold[available_engineered_columns].head(20))

## Part 8: Load California Unified School District Boundaries

The California Department of Education's 2025–26 School District Areas layer is queried through its official ArcGIS GeoJSON endpoint. The API request filters to `DistrictType = 'Unified'` before download, reducing file size and avoiding overlapping elementary/high-school district systems.

The boundary layer is requested in EPSG:4326 so it matches property longitude and latitude coordinates.

In [ ]:
DISTRICT_API = (
    "https://services3.arcgis.com/fdvHcZVgB2QSRNkL/arcgis/rest/services/"
    "DistrictAreas2526/FeatureServer/0/query"
)

district_params = {
    "where": "DistrictType = 'Unified'",
    "outFields": "DistrictName,DistrictType,CountyName,CDCode",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson",
}

response = requests.get(DISTRICT_API, params=district_params, timeout=180)
response.raise_for_status()
district_geojson = response.json()

if "features" not in district_geojson:
    raise ValueError(f"Unexpected district API response: {district_geojson}")

unified_districts = gpd.GeoDataFrame.from_features(
    district_geojson["features"],
    crs="EPSG:4326",
)

required_district_columns = {"DistrictName", "DistrictType", "geometry"}
missing_district_columns = required_district_columns.difference(unified_districts.columns)
if missing_district_columns:
    raise KeyError(f"District data is missing expected columns: {sorted(missing_district_columns)}")

unified_districts = unified_districts[
    ["DistrictName", "DistrictType", "CountyName", "CDCode", "geometry"]
].copy()
unified_districts["geometry"] = unified_districts.geometry.make_valid()

print("Unified district polygons loaded:", len(unified_districts))
print("District CRS:", unified_districts.crs)
display(unified_districts.drop(columns="geometry").head())

## Part 9: Spatially Join Properties to Unified School Districts

Valid California coordinates are converted to geographic points and matched to the district polygon that contains each point. Processing occurs in chunks to reduce peak memory use for the large MLS datasets.

Records with missing or implausible coordinates remain in the dataset with a missing `DistrictName`. Exact boundary points may also remain unmatched under the `within` rule and should be reviewed separately if needed.

In [ ]:
def add_unified_school_district(df, districts, chunk_size=100_000):
    required_coordinates = {"Latitude", "Longitude"}
    missing_coordinates = required_coordinates.difference(df.columns)
    if missing_coordinates:
        raise KeyError(f"Property data is missing coordinate columns: {sorted(missing_coordinates)}")

    enriched = df.copy()
    enriched["Latitude"] = pd.to_numeric(enriched["Latitude"], errors="coerce")
    enriched["Longitude"] = pd.to_numeric(enriched["Longitude"], errors="coerce")
    enriched["DistrictName"] = pd.Series(pd.NA, index=enriched.index, dtype="string")

    valid_mask = (
        enriched["Latitude"].between(32, 42)
        & enriched["Longitude"].between(-125, -114)
    )
    valid_indices = enriched.index[valid_mask]

    for start in range(0, len(valid_indices), chunk_size):
        chunk_indices = valid_indices[start:start + chunk_size]
        chunk = enriched.loc[chunk_indices, ["Longitude", "Latitude"]].copy()

        points = gpd.GeoDataFrame(
            chunk,
            geometry=gpd.points_from_xy(chunk["Longitude"], chunk["Latitude"]),
            crs="EPSG:4326",
        )

        joined = gpd.sjoin(
            points,
            districts[["DistrictName", "geometry"]],
            how="left",
            predicate="within",
        )
        joined = joined.loc[~joined.index.duplicated(keep="first")]
        enriched.loc[joined.index, "DistrictName"] = joined["DistrictName"].astype("string")

        processed = min(start + chunk_size, len(valid_indices))
        print(f"Mapped {processed:,} of {len(valid_indices):,} valid coordinate records")

    return enriched

### Run and Validate the Spatial Join

The sample test provides a fast confirmation before processing both complete datasets. If the sample is successful, run the full mapping cells.

In [ ]:
sold_mapping_test = add_unified_school_district(
    sold.head(1_000),
    unified_districts,
    chunk_size=1_000,
)

display(sold_mapping_test[["Latitude", "Longitude", "DistrictName"]].head(20))
print("Test match rate:", sold_mapping_test["DistrictName"].notna().mean())

In [ ]:
sold = add_unified_school_district(sold, unified_districts)
listings = add_unified_school_district(listings, unified_districts)

district_match_summary = pd.DataFrame({
    "Dataset": ["Sold", "Listings"],
    "TotalRecords": [len(sold), len(listings)],
    "MatchedRecords": [sold["DistrictName"].notna().sum(), listings["DistrictName"].notna().sum()],
})
district_match_summary["UnmatchedRecords"] = (
    district_match_summary["TotalRecords"] - district_match_summary["MatchedRecords"]
)
district_match_summary["MatchRate"] = (
    district_match_summary["MatchedRecords"] / district_match_summary["TotalRecords"]
)
display(district_match_summary)

## Part 10: Property Subtype Summary

Compare transaction volume, median pricing, market speed, and negotiation outcomes across residential property categories. Median price measures are used because housing-price distributions are strongly right-skewed.

In [ ]:
property_subtype_summary = (
    sold.groupby("PropertySubType", dropna=False)
    .agg(
        ClosedSales=("ListingKey", "count"),
        MedianClosePrice=("ClosePrice", "median"),
        MedianPricePerSqFt=("PricePerSqFt", "median"),
        MedianDaysOnMarket=("DaysOnMarket", "median"),
        MedianCloseToOriginalListRatio=("CloseToOriginalListRatio", "median"),
    )
    .reset_index()
    .sort_values("ClosedSales", ascending=False)
)
display(property_subtype_summary.head(20))

## Part 11: County Market Summary

Create a geographic market comparison for Tableau. Mortgage rate is included when an available rate field is detected.

In [ ]:
county_aggregations = {
    "ClosedSales": ("ListingKey", "count"),
    "MedianClosePrice": ("ClosePrice", "median"),
    "MedianPricePerSqFt": ("PricePerSqFt", "median"),
    "MedianDaysOnMarket": ("DaysOnMarket", "median"),
    "MedianCloseToOriginalListRatio": ("CloseToOriginalListRatio", "median"),
}

rate_candidates = [column for column in sold.columns if "rate" in column.lower()]
preferred_rate = (
    "rate_30yr_fixed"
    if "rate_30yr_fixed" in sold.columns
    else (rate_candidates[0] if rate_candidates else None)
)
if preferred_rate:
    sold[preferred_rate] = pd.to_numeric(sold[preferred_rate], errors="coerce")
    county_aggregations["AverageMortgageRate"] = (preferred_rate, "mean")

county_summary = (
    sold.groupby("CountyOrParish", dropna=False)
    .agg(**county_aggregations)
    .reset_index()
    .sort_values("ClosedSales", ascending=False)
)
display(county_summary.head(20))

## Part 12: Unified School District Market Summary

Use the newly enriched `DistrictName` field to compare market activity and pricing across matched Unified School Districts. Unmatched properties are excluded from this summary but remain in the detailed datasets.

In [ ]:
school_district_summary = (
    sold.dropna(subset=["DistrictName"])
    .groupby("DistrictName")
    .agg(
        ClosedSales=("ListingKey", "count"),
        MedianClosePrice=("ClosePrice", "median"),
        MedianPricePerSqFt=("PricePerSqFt", "median"),
        MedianDaysOnMarket=("DaysOnMarket", "median"),
        MedianCloseToOriginalListRatio=("CloseToOriginalListRatio", "median"),
    )
    .reset_index()
    .sort_values("ClosedSales", ascending=False)
)
display(school_district_summary.head(20))

## Part 13: Listing Office Competitive Summary

Summarize listing offices by transaction volume, total sales value, median sale price, and median market time. Office names are treated as source values; capitalization variants should be standardized in a future entity-resolution step if required.

In [ ]:
office_summary = (
    sold.groupby("ListOfficeName", dropna=False)
    .agg(
        ClosedSales=("ListingKey", "count"),
        TotalSalesVolume=("ClosePrice", "sum"),
        MedianClosePrice=("ClosePrice", "median"),
        MedianDaysOnMarket=("DaysOnMarket", "median"),
    )
    .reset_index()
    .sort_values("TotalSalesVolume", ascending=False)
)
display(office_summary.head(20))

## Part 14: Final Quality Assurance

Confirm that engineered columns exist, row counts remain unchanged, district enrichment is available, and the primary export tables contain records.

In [ ]:
required_sold_outputs = {
    "PriceRatio", "CloseToOriginalListRatio", "PricePerSqFt",
    "Year", "Month", "YrMo", "ListingToContractDaysValid",
    "ContractToCloseDaysValid", "DistrictName",
}
required_listing_outputs = {"ListingYear", "ListingMonth", "ListingYrMo", "DistrictName"}

assert required_sold_outputs.issubset(sold.columns), "Sold output is missing engineered fields."
assert required_listing_outputs.issubset(listings.columns), "Listings output is missing engineered fields."
assert len(sold) == load_summary.loc[load_summary["Dataset"].eq("Sold"), "Rows"].iloc[0]
assert len(listings) == load_summary.loc[load_summary["Dataset"].eq("Listings"), "Rows"].iloc[0]
assert not property_subtype_summary.empty
assert not county_summary.empty
assert not school_district_summary.empty
assert not office_summary.empty

final_qa_summary = pd.DataFrame({
    "Dataset": ["Listings", "Sold"],
    "Rows": [len(listings), len(sold)],
    "Columns": [listings.shape[1], sold.shape[1]],
    "DistrictMatchRate": [
        listings["DistrictName"].notna().mean(),
        sold["DistrictName"].notna().mean(),
    ],
})
display(final_qa_summary)
print("Final QA checks passed.")

## Part 15: Export Week 6 Deliverables

Export enriched detail datasets, segment summaries, spatial-match QA, and engineered-metric QA. These files become the standard Week 6 inputs for the remainder of the project.

In [ ]:
export_objects = {
    "feature_engineered_sold.csv": sold,
    "feature_engineered_listings.csv": listings,
    "property_subtype_summary.csv": property_subtype_summary,
    "county_summary.csv": county_summary,
    "school_district_summary.csv": school_district_summary,
    "listing_office_summary.csv": office_summary,
    "school_district_match_summary.csv": district_match_summary,
    "engineered_metric_quality_summary.csv": quality_summary,
}

export_log = []
for filename, dataframe in export_objects.items():
    destination = OUTPUT_DIR / filename
    dataframe.to_csv(destination, index=False)
    export_log.append({
        "File": filename,
        "Rows": len(dataframe),
        "Columns": dataframe.shape[1],
        "Path": str(destination.resolve()),
    })

export_log = pd.DataFrame(export_log)
display(export_log)

# Week 6 Summary

Week 6 produced a complete feature-engineering and geographic-enrichment workflow.

## Completed Work

- Created Price Ratio, Close-to-Original-List Ratio, and Price Per Square Foot
- Created sold and listing time-series fields
- Calculated listing-to-contract and contract-to-close durations
- Preserved raw metrics and added transparent validation flags
- Mapped valid property coordinates to California Unified School Districts
- Generated summaries by property subtype, county, school district, and listing office
- Exported enriched datasets and QA tables for Tableau and downstream analysis

## Important Data-Quality Notes

- Extreme price and timeline values are flagged rather than silently removed
- Missing or invalid coordinates remain unmatched but are retained
- Exact district-boundary points may require separate review
- Listing-office capitalization and naming variants remain a future standardization opportunity

The exported Week 6 datasets are ready to be used throughout the remainder of the project.

In [ ]:
import pandas as pd

sold = pd.read_csv(
    "../outputs/week7_sold_flagged.csv",
    low_memory=False
)

sold["CloseDate"] = pd.to_datetime(
    sold["CloseDate"],
    errors="coerce"
)

print(
    sold.groupby(
        sold["CloseDate"].dt.to_period("M")
    ).size().loc[
        "2024-01":"2024-03"
    ]
)

print(
    "Duplicate ListingKey:",
    sold["ListingKey"].duplicated().sum()
)